In [ ]:
from time import perf_counter

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.metrics.pairwise import linear_kernel, rbf_kernel
from sklearn.svm import SVC

from svm import get_default_params, train_linear_svm, train_rbf_svm

FIT_REPEATS = 5
KERNEL_REPEATS = 20
PREDICT_REPEATS = 100
POSITIVE_LABEL = 1  # Set to the positive class in your labels.


In [ ]:
# Placeholders
X_train = None
Y_train = None
X_test = None
Y_test = None

if any(value is None for value in (X_train, Y_train, X_test, Y_test)):
    raise ValueError("Provide X_train, Y_train, X_test, and Y_test before running the evaluation.")


In [ ]:
# Measure how long an operation takes (median)
def benchmark(call, repeats):
    """Warm up, then return the final result and median elapsed seconds."""
    call()
    durations = []
    result = None
    for _ in range(repeats):
        start = perf_counter()
        result = call()
        durations.append(perf_counter() - start)
    return result, float(np.median(durations))


def resolve_gamma(gamma, X):
    if gamma == "scale":
        variance = X.var()
        return 1.0 / (X.shape[1] * variance) if variance else 1.0
    if gamma == "auto":
        return 1.0 / X.shape[1]
    return float(gamma)


params = get_default_params()
models = {}
quality_rows = []
timing_rows = []

for name in ("linear", "rbf"):
    settings = params[name]
    if name == "linear":
        model, fit_seconds = benchmark(
            lambda: train_linear_svm(X_train, Y_train, **settings), FIT_REPEATS
        )
    else:
        model, fit_seconds = benchmark(
            lambda: train_rbf_svm(X_train, Y_train, **settings), FIT_REPEATS
        )
    models[name] = model
    predictions, predict_seconds = benchmark(
        lambda: model.predict(X_test), PREDICT_REPEATS
    )
    precision, recall, f1, _ = precision_recall_fscore_support(
        Y_test, predictions, average="binary", pos_label=POSITIVE_LABEL,
        zero_division=0,
    )
    quality_rows.append({
        "Model": name,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "Test accuracy": accuracy_score(Y_test, predictions),
    })

    if name == "linear":
        kernel = linear_kernel
    else:
        gamma = resolve_gamma(settings["gamma"], X_train)
        kernel = lambda A, B: rbf_kernel(A, B, gamma=gamma)

    K_train, train_kernel_seconds = benchmark(
        lambda: kernel(X_train, X_train), KERNEL_REPEATS
    )
    K_test, test_kernel_seconds = benchmark(
        lambda: kernel(X_test, X_train), KERNEL_REPEATS
    )
    precomputed_model, precomputed_fit_seconds = benchmark(
        lambda: SVC(kernel="precomputed", C=settings["C"]).fit(K_train, Y_train),
        FIT_REPEATS,
    )
    np.testing.assert_array_equal(precomputed_model.predict(K_test), predictions)
    np.testing.assert_allclose(
        precomputed_model.decision_function(K_test),
        model.decision_function(X_test), atol=1e-5, rtol=1e-5,
    )
    timing_rows.append({
        "Model": name,
        "SVM training (ms)": fit_seconds * 1000,
        "Train kernel (ms)": train_kernel_seconds * 1000,
        "Test/train kernel (ms)": test_kernel_seconds * 1000,
        "Precomputed SVM fit (ms)": precomputed_fit_seconds * 1000,
        "Inference batch (ms)": predict_seconds * 1000,
        "Inference/sample (us)": predict_seconds * 1_000_000 / len(Y_test),
    })

quality = pd.DataFrame(quality_rows).set_index("Model")
timings = pd.DataFrame(timing_rows).set_index("Model")
print("Repository and precomputed-kernel predictions and decision scores agree.")


In [ ]:
print("Classification metrics:")
print(quality.to_string(float_format=lambda value: f"{value:.4f}"))
print("\nTiming metrics (median after warm-up):")
print(timings.to_string(float_format=lambda value: f"{value:.4f}"))
print(f"\nRepeats: training={FIT_REPEATS}, kernels={KERNEL_REPEATS}, inference={PREDICT_REPEATS}")


In [ ]:
K_linear = linear_kernel(X_train, X_train)
n_train = K_linear.shape[0]

plt.figure(figsize=(7, 6))
plt.imshow(K_linear, cmap="viridis")
plt.colorbar(label="Linear kernel value")
plt.xlabel("Training sample index")
plt.ylabel("Training sample index")
plt.title(f"Linear Kernel Matrix ({n_train} ? {n_train})")
plt.tight_layout()
plt.show()

eigenvalues = np.linalg.eigvalsh(K_linear)
off_diagonal = K_linear[np.triu_indices(n_train, k=1)]
print("Minimum eigenvalue:", eigenvalues.min())
print("Maximum eigenvalue:", eigenvalues.max())
print("Mean off-diagonal similarity:", off_diagonal.mean() if off_diagonal.size else float("nan"))


In [ ]:
gamma = resolve_gamma(params["rbf"]["gamma"], X_train)
K_rbf = rbf_kernel(X_train, X_train, gamma=gamma)
n_train = K_rbf.shape[0]

plt.figure(figsize=(7, 6))
plt.imshow(K_rbf, cmap="viridis", vmin=0, vmax=1)
plt.colorbar(label="RBF kernel similarity")
plt.xlabel("Training sample index")
plt.ylabel("Training sample index")
plt.title(f"RBF Kernel Matrix ({n_train} ? {n_train})")
plt.tight_layout()
plt.show()

eigenvalues = np.linalg.eigvalsh(K_rbf)
off_diagonal = K_rbf[np.triu_indices(n_train, k=1)]
print("Minimum eigenvalue:", eigenvalues.min())
print("Maximum eigenvalue:", eigenvalues.max())
print("Mean off-diagonal similarity:", off_diagonal.mean() if off_diagonal.size else float("nan"))
